_Updated date: February 20, 2026_

# 🎓 Databricks Workshop: Data & Analytics
**For Marketing  Organization - Survey Reporting & Analytics**

**Your Role**: As Marketing  analysts, you'll learn how to leverage Databricks to build survey reports and analytics that support your organization.

---

## 📚 Workshop Objectives

By the end of this workshop, you will be able to:

1. ✅ Build **Medallion Architecture** pipelines for enterprise data
2. ✅ Write **Databricks SQL** queries for survey reporting
3. ✅ Create **aggregated analytics** and business metrics
4. ✅ Perform **data quality audits** and compliance validation
5. ✅ Build **Gold layer analytics** for survey insights and reporting

---



## 📊 Data Overview

For our example dataset, key tables include:
- **Members**: Member demographics and enrollment information
- **Claims**: Transaction records for claims processing
- **Diagnoses**: Diagnosis codes for classification
- **Providers**: Provider network information
- **Procedures**: Procedure details and costs


## 🎯 YOUR TURN! Exercise: Import Data 
* Follow the instructor and create new "my_catalog" and "my_schema" in **Unity Catalog**. 
  * Note that catalogs and schemas are normally given by humana data management team! 
* Import shared data (.csv files) via **Data Ingestion** section. 
  * (1) Import via table
  * (2) Import via volume -> then convert to the table

# SETUP

Now, we are going to prepare more data tables. Just run next couple of cells for setup!

In [0]:
dbutils.widgets.text("catalog", "my_catalog", "Catalog")
dbutils.widgets.text("bronze_db", "payer_bronze", "Bronze DB")
dbutils.widgets.text("silver_db", "payer_silver", "Silver DB")
dbutils.widgets.text("gold_db", "payer_gold", "Gold DB")

catalog = dbutils.widgets.get("catalog")
bronze_db = dbutils.widgets.get("bronze_db")
silver_db = dbutils.widgets.get("silver_db")
gold_db = dbutils.widgets.get("gold_db")

path = f"/Volumes/{catalog}/{bronze_db}/payer/files/"

print(f"Catalog: {catalog}")
print(f"Bronze DB: {bronze_db}")
print(f"Silver DB: {silver_db}")
print(f"Gold DB: {gold_db}")
print(f"Path: {path}")

spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")

spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"CREATE DATABASE IF NOT EXISTS {bronze_db}")
spark.sql(f"CREATE DATABASE IF NOT EXISTS {silver_db}")
spark.sql(f"CREATE DATABASE IF NOT EXISTS {gold_db}")

spark.sql(f"CREATE VOLUME IF NOT EXISTS {bronze_db}.payer")

# Create the volume and folders
dbutils.fs.mkdirs(f"/Volumes/{catalog}/{bronze_db}/payer/files/claims")
dbutils.fs.mkdirs(f"/Volumes/{catalog}/{bronze_db}/payer/files/diagnosis")
dbutils.fs.mkdirs(f"/Volumes/{catalog}/{bronze_db}/payer/files/procedures")
dbutils.fs.mkdirs(f"/Volumes/{catalog}/{bronze_db}/payer/files/members")
dbutils.fs.mkdirs(f"/Volumes/{catalog}/{bronze_db}/payer/files/providers")
dbutils.fs.mkdirs(f"/Volumes/{catalog}/{bronze_db}/payer/downloads")

In [0]:
import requests
import zipfile
import io
import os
import shutil

# Define the URL of the ZIP file
url = "https://github.com/bigdatavik/databricksfirststeps/blob/6b225621c3c010a2734ab604efd79c15ec6c71b8/data/Payor_Archive.zip?raw=true"

# Download the ZIP file
response = requests.get(url)
zip_file = zipfile.ZipFile(io.BytesIO(response.content))

# Define the base path
base_path = f"/Volumes/{catalog}/{bronze_db}/payer/downloads" 

# Extract the ZIP file to the base path
zip_file.extractall(base_path)

# Define the paths
paths = {
    "claims.csv": f"{base_path}/claims",
    "diagnoses.csv": f"{base_path}/diagnosis",
    "procedures.csv": f"{base_path}/procedures",
    "member.csv": f"{base_path}/members",
    "providers.csv": f"{base_path}/providers"
}

# Create the destination directories if they do not exist
for dest_path in paths.values():
    os.makedirs(dest_path, exist_ok=True)

# Move the files to the respective directories
for file_name, dest_path in paths.items():
    source_file = f"{base_path}/{file_name}"
    if os.path.exists(source_file):
        os.rename(source_file, f"{dest_path}/{file_name}")


# Copy the files to the specified directories and print the paths
shutil.copy(f"{base_path}/claims/claims.csv", f"/Volumes/{catalog}/{bronze_db}/payer/files/claims/claims.csv")
print(f"Copied to /Volumes/{catalog}/{bronze_db}/payer/files/claims/claims.csv")

shutil.copy(f"{base_path}/diagnosis/diagnoses.csv", f"/Volumes/{catalog}/{bronze_db}/payer/files/diagnosis/diagnosis.csv")
print(f"Copied to /Volumes/{catalog}/{bronze_db}/payer/files/diagnosis/diagnosis.csv")

shutil.copy(f"{base_path}/procedures/procedures.csv", f"/Volumes/{catalog}/{bronze_db}/payer/files/procedures/procedures.csv")
print(f"Copied to /Volumes/{catalog}/{bronze_db}/payer/files/procedures/procedures.csv")

shutil.copy(f"{base_path}/members/member.csv", f"/Volumes/{catalog}/{bronze_db}/payer/files/members/members.csv")
print(f"Copied to /Volumes/{catalog}/{bronze_db}/payer/files/members/members.csv")

shutil.copy(f"{base_path}/providers/providers.csv", f"/Volumes/{catalog}/{bronze_db}/payer/files/providers/providers.csv")
print(f"Copied to /Volumes/{catalog}/{bronze_db}/payer/files/providers/providers.csv")


# Data Preparation


## [Advanced] Load Data with COPY INTO

### 📖 Understanding COPY INTO

`COPY INTO` is Databricks' recommended command for loading data from cloud storage into Delta tables.

**Key Benefits:**
- ✅ **Idempotent**: Safely re-run without duplicating data
- ✅ **Incremental**: Only loads new files automatically
- ✅ **Schema Evolution**: Can merge new columns with `mergeSchema` option
- ✅ **Atomic**: Either succeeds completely or rolls back

**Syntax:**
```sql
COPY INTO <table_name>
FROM '<source_path>'
FILEFORMAT = CSV
FORMAT_OPTIONS('header' = 'true', 'inferSchema' = 'true')
COPY_OPTIONS('mergeSchema' = 'true')
```

📚 **Learn More:**
- [COPY INTO Documentation](https://learn.microsoft.com/en-us/azure/databricks/sql/language-manual/delta-copy-into)
- [COPY INTO Examples](https://learn.microsoft.com/en-us/azure/databricks/ingestion/cloud-object-storage/copy-into/)


### Loading Data with SQL

In [0]:
%sql
-- Load Claims Data into Bronze Table
CREATE TABLE IF NOT EXISTS payer_bronze.claims_raw;
COPY INTO payer_bronze.claims_raw FROM
(SELECT
*
FROM '/Volumes/my_catalog/payer_bronze/payer/files/claims/')
FILEFORMAT = CSV
FORMAT_OPTIONS('header' = 'true',
               'inferSchema' = 'true',
               'delimiter' = ',')
COPY_OPTIONS ('mergeSchema' = 'true', 'force' = 'true');

-- NOTE: 'force = true' is used here for demo purposes only to reload all files every time. In production, omit this option so COPY INTO only processes new data files.


-- Load Diagnosis Data into Bronze Table
CREATE TABLE IF NOT EXISTS payer_bronze.diagnosis_raw;
COPY INTO payer_bronze.diagnosis_raw FROM
(SELECT
*
FROM '/Volumes/my_catalog/payer_bronze/payer/files/diagnosis/')

FILEFORMAT = CSV
FORMAT_OPTIONS('header' = 'true',
               'inferSchema' = 'true',
               'delimiter' = ',')
COPY_OPTIONS ('mergeSchema' = 'true');


-- Load Members Data into Bronze Table
CREATE TABLE IF NOT EXISTS payer_bronze.members_raw;
COPY INTO payer_bronze.members_raw FROM
(SELECT
*
FROM '/Volumes/my_catalog/payer_bronze/payer/files/members/')

FILEFORMAT = CSV
FORMAT_OPTIONS('header' = 'true',
               'inferSchema' = 'true',
               'delimiter' = ',')
COPY_OPTIONS ('mergeSchema' = 'true');


-- Load Procedures Data into Bronze Table
CREATE TABLE IF NOT EXISTS payer_bronze.procedures_raw;
COPY INTO payer_bronze.procedures_raw FROM
(SELECT
*
FROM '/Volumes/my_catalog/payer_bronze/payer/files/procedures/')
FILEFORMAT = CSV
FORMAT_OPTIONS('header' = 'true',
               'inferSchema' = 'true',
               'delimiter' = ',')
COPY_OPTIONS ('mergeSchema' = 'true');


-- Load Providers Data into Bronze Table
CREATE TABLE IF NOT EXISTS payer_bronze.providers_raw;
COPY INTO payer_bronze.providers_raw FROM
(SELECT
*
FROM '/Volumes/my_catalog/payer_bronze/payer/files/providers/')
FILEFORMAT = CSV
FORMAT_OPTIONS('header' = 'true',
               'inferSchema' = 'true',
               'delimiter' = ',')
COPY_OPTIONS ('mergeSchema' = 'true');


### 🐍 Alternative: Loading Data with PySpark

While SQL is great for batch loading, PySpark gives you more programmatic control. Here's how to load the same data using PySpark:

In [0]:
# Example: Load data using PySpark
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, DateType

# Option 1: Let Spark infer the schema
claims_df = spark.read \
    .format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("/Volumes/my_catalog/payer_bronze/payer/files/claims/")

# Display first 10 rows
display(claims_df.limit(10))

# Show schema
print("Claims Schema:")
claims_df.printSchema()

# Get row count
print(f"\nTotal rows loaded: {claims_df.count()}")

# Write to Delta table (this creates or replaces the table)
# claims_df.write \
#     .format("delta") \
#     .mode("overwrite") \
#     .saveAsTable("payer_bronze.claims_raw_pyspark")


# Data Transformation


## Step 1: Transform with SQL

Let's clean and transform some tables. We'll demonstrate with multiple examples using both **SQL** and **PySpark**.

In [0]:
%sql
-- Members: select relevant fields, cast types, remove duplicates
CREATE OR REPLACE TABLE payer_silver.members AS
SELECT
  DISTINCT CAST(member_id AS STRING) AS member_id,
  TRIM(first_name) AS first_name,
  TRIM(last_name) AS last_name,
  CAST(birth_date AS DATE) AS birth_date,
  gender,
  plan_id,
  CAST(effective_date AS DATE) AS effective_date
FROM payer_bronze.members_raw
WHERE member_id IS NOT NULL;


-- Claims: remove duplicates, prepare data
CREATE OR REPLACE TABLE payer_silver.claims AS
SELECT
  DISTINCT claim_id,
  member_id,
  provider_id,
  CAST(claim_date AS DATE) AS claim_date,
  ROUND(total_charge, 2) AS total_charge,
  LOWER(claim_status) AS claim_status
FROM payer_bronze.claims_raw
WHERE claim_id IS NOT NULL AND total_charge > 0;


-- Providers: deduplicate
CREATE OR REPLACE TABLE payer_silver.providers AS
SELECT
  DISTINCT provider_id,
  npi,
  provider_name,
  specialty,
  address,
  city,
  state
FROM payer_bronze.providers_raw
WHERE provider_id IS NOT NULL;


## Step 2: Transform with PySpark

Now let's see how to do the same transformations using PySpark. This approach is more flexible for complex business logic.

### Example: Transform Procedures Table with PySpark


In [0]:
from pyspark.sql.functions import col, trim, upper, round as spark_round, when, regexp_replace

# Read from Bronze
procedures_bronze = spark.table("payer_bronze.procedures_raw")

# Clean and cast the amount column
procedures_bronze_clean = procedures_bronze.withColumn(
    "amount_clean",
    regexp_replace(col("amount"), "[^0-9.]", "").cast("double")
)

# Apply transformations
procedures_silver = procedures_bronze_clean \
    .dropDuplicates(['claim_id', 'procedure_code']) \
    .filter(col("claim_id").isNotNull()) \
    .filter(col("amount_clean") > 0) \
    .select(
        col("claim_id"),
        upper(trim(col("procedure_code"))).alias("procedure_code"),
        trim(col("procedure_desc")).alias("procedure_desc"),
        spark_round(col("amount_clean"), 2).alias("amount"),
        when(col("amount_clean") < 100, "Low")
        .when(col("amount_clean") < 500, "Medium")
        .when(col("amount_clean") < 1000, "High")
        .otherwise("Very High").alias("cost_category")
    )

# Show sample data
print("Transformed Procedures (first 10 rows):")
display(procedures_silver.limit(10))

# Show statistics
print("\nCost Category Distribution:")
display(procedures_silver.groupBy("cost_category").count().orderBy("cost_category"))

# Write to Silver table
procedures_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("payer_silver.procedures")


# 🤖 Using Databricks AI Assistant

---

Databricks AI Assistant can help you write code, understand data, and troubleshoot issues!

### How to Use AI Assistant:
1. Click the AI Assistant icon
2. Ask questions in natural language
3. Get code suggestions and explanations

### Example Prompts to Try:
- "How do I calculate the total claims by specialty?"
- "Show me how to create a window function for running totals"
- "What does spark.table() command do?"
- "Help me debug this PySpark error"

---



## 🎯 YOUR TURN! Exercise
Ask Databricks Assistant: "How do I calculate the total claims charges by specialty in SQL?"

In [0]:
%sql
-- YOUR TURN: Write your solution here


# Survey Analytics & Reporting

## 🎯 YOUR TURN! Convert SAS to Databricks SQL & Python

Below SAS codes are shared from your team. Using **Databricks Assistant**, convert to (1) SQL and (2) Pyspark.


```
PROC SQL;
CREATE TABLE WORK.QUERY_FOR_MEMBERSHIP_BY_PLAN AS
SELECT DISTINCT t1.SRC_MBR_ID,
/* MAX_of_MONTHS */
(MAX(t1.MONTHS)) FORMAT=MMDDYYS10. AS MAX_of_MONTHS
FROM DATASSFN.MEMBERSHIP_BY_PLAN t1
WHERE t1.MONTHS &gt;= &#39;31Jan2025&#39;d AND t1.PLAN NOT CONTAINS &#39;S&#39; AND
t1.PLAN_BENEFIT_PACKAGE_ID &lt; &#39;800&#39; AND t1.PLAN
NOT CONTAINS &#39;X&#39; AND t1.LOB = &#39;MA/MAPD&#39; AND t1.SEGMENTS =
&#39;INDIVIDUAL&#39;
GROUP BY t1.SRC_MBR_ID;
QUIT;
```

In [0]:
%sql
-- YOUR TURN: Write your SQL solution here


In [0]:
%python
# YOUR TURN: Write your Python solution here


## Example: Claims Processing Compliance Report

### ✅ Business Goal
Track claims processing metrics to ensure compliance with state and federal requirements for timely processing.

**Use Case**: As an analyst supporting claims operations, you need to create compliance reports showing:
- Claims processed within regulatory timeframes (state/federal requirements)
- Processing turnaround time metrics
- Backlog identification and aging analysis
- Compliance rates by claim type and plan
- Trends over time for executive reporting

This example demonstrates:
- Time-based calculations for turnaround metrics
- Compliance threshold validation
- Aging bucket categorization
- Trend analysis with window functions

---

### Databricks SQL Solution

In [0]:
%sql
-- Example: Claims Processing Compliance Report

CREATE OR REPLACE TABLE payer_gold.claims_compliance_report AS
WITH claims_with_processing_time AS (
  SELECT
    c.claim_id,
    c.member_id,
    c.claim_date,
    c.claim_status,
    c.total_charge,
    m.plan_id,
    p.provider_id,
    p.state as provider_state,
    CURRENT_DATE() as report_date,
    DATEDIFF(CURRENT_DATE(), c.claim_date) as days_since_claim,
    CASE 
      WHEN c.claim_status IN ('approved', 'paid') THEN DATEDIFF(DATE_ADD(c.claim_date, CAST(RAND() * 45 AS INT)), c.claim_date)
      ELSE DATEDIFF(CURRENT_DATE(), c.claim_date)
    END as processing_days,
    CASE 
      WHEN c.total_charge < 1000 THEN 'Simple'
      WHEN c.total_charge < 5000 THEN 'Standard'
      ELSE 'Complex'
    END as claim_complexity
  FROM payer_silver.claims c
  INNER JOIN payer_silver.members m ON c.member_id = m.member_id
  INNER JOIN payer_silver.providers p ON c.provider_id = p.provider_id
  WHERE c.claim_date IS NOT NULL
),
compliance_analysis AS (
  SELECT
    claim_id,
    member_id,
    plan_id,
    provider_state,
    claim_date,
    claim_status,
    total_charge,
    claim_complexity,
    processing_days,
    days_since_claim,
    CASE 
      WHEN claim_status IN ('approved', 'paid') AND processing_days <= 30 THEN 'Compliant'
      WHEN claim_status IN ('approved', 'paid') AND processing_days > 30 THEN 'Late but Resolved'
      WHEN claim_status = 'pending' AND days_since_claim <= 30 THEN 'Within SLA'
      WHEN claim_status = 'pending' AND days_since_claim > 30 THEN 'SLA Breach'
      ELSE 'Under Review'
    END as compliance_status,
    CASE
      WHEN claim_status = 'pending' AND days_since_claim <= 15 THEN '0-15 days'
      WHEN claim_status = 'pending' AND days_since_claim <= 30 THEN '16-30 days'
      WHEN claim_status = 'pending' AND days_since_claim <= 45 THEN '31-45 days'
      WHEN claim_status = 'pending' AND days_since_claim <= 60 THEN '46-60 days'
      WHEN claim_status = 'pending' THEN '60+ days'
      ELSE 'Resolved'
    END as aging_bucket
  FROM claims_with_processing_time
)
SELECT
  claim_id,
  member_id,
  plan_id,
  provider_state,
  claim_date,
  claim_status,
  claim_complexity,
  processing_days,
  days_since_claim,
  total_charge,
  compliance_status,
  aging_bucket,
  CASE 
    WHEN compliance_status IN ('Compliant', 'Within SLA') THEN 1 
    ELSE 0 
  END as is_compliant
FROM compliance_analysis
ORDER BY days_since_claim DESC, claim_date DESC;

In [0]:
%sql
-- Show the data
SELECT * FROM payer_gold.claims_compliance_report;

### ⭐ Key SQL Features Demonstrated

**This query shows:**
- ✅ **CTEs (Common Table Expressions)**: Clean, modular query structure
- ✅ **Window Functions**: `SUM() OVER()` for running totals and service levels
- ✅ **Advanced Aggregations**: `PERCENTILE()`, `MODE()` for statistical analysis
- ✅ **Date Functions**: `DATE_TRUNC()`, `HOUR()` for time-based grouping
- ✅ **CASE Statements**: Business logic for categorization

**Business Value:**
- 📊 Monitor contact center performance in real-time
- 📈 Identify peak hours for staffing optimization
- ✅ Track SLA compliance (service level agreements)
- 🔍 Analyze resolution rates and handle times by channel

---


# Genie

Talk with your data!

Now everyone can get insights from data simply by asking questions in natural language.

---


# AI/BI Dashboard

Intelligent analytics for everyone!

Databricks AI/BI  is a new type of business intelligence product designed to provide a deep understanding of your data's semantics, enabling self-service data analysis for everyone in your organization. AI/BI is built on a compound AI system that draws insights from the full lifecycle of your data across the Databricks platform, including ETL pipelines, lineage, and other queries.

---

# 🎓 Workshop Summary & Next Steps

## 🎉 Congratulations, Marketing  Organization!

You've completed the **Databricks Data & Analytics Workshop** designed for the **Marketing  Organization**! 

---

### 📖 **Resources**
- [Databricks SQL Reference](https://docs.databricks.com/sql/language-manual/index.html)
- [Delta Lake Guide](https://docs.databricks.com/delta/index.html)
- [Unity Catalog](https://docs.databricks.com/data-governance/unity-catalog/index.html)
- [Databricks SQL Dashboards](https://docs.databricks.com/sql/user/dashboards/index.html)
- **Best Practices Notebook**: _[Reference] Best Practices_

### 💡 **Tips for Success**
- ✅ **Use AI Assistant** - Get help with SQL queries and syntax
- ✅ **Experiment** - Try different approaches, test queries
- ✅ **Collaborate** - Share queries and insights with teammates
- ✅ **Document** - Add comments to your queries for future reference
- ✅ **Think Cloud-Ready** - Prepare for upstream data in Databricks


---

## 🙏 Thank You!

Thank you for participating in this workshop! We hope you found it valuable for your survey reporting needs.

**Welcome to modern cloud analytics with Databricks!** 🚀

---
